# Phase 5: VLM Dataset Preparation

**Objectif:** Préparer le dataset pour l'entraînement de Vision-Language Models (Qwen2.5-VL)

**Dataset source:**
- 30,491 images traitées (99.9% succès)
- 448×448 pixels, normalisées
- 5 classes: CBB, CBSD, CGM, CMD, Healthy

**Tâches:**
1. Créer splits train/val/test stratifiés
2. Générer format Qwen2.5-VL (JSONL)
3. Créer manifestes et statistiques
4. Valider l'intégrité du dataset

**Outputs:**
- `vlm_dataset/train.jsonl` - Données d'entraînement
- `vlm_dataset/val.jsonl` - Données de validation
- `vlm_dataset/test.jsonl` - Données de test
- `vlm_dataset/dataset_info.json` - Métadonnées et statistiques
- `vlm_dataset/class_mapping.json` - Mapping des classes

**Splits proposés:**
- Train: 80% (~24,393 images)
- Val: 10% (~3,049 images)
- Test: 10% (~3,049 images)

## 1. Setup & Imports

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import time
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split
import shutil
from datetime import datetime

print("✓ Imports successful")

✓ Imports successful


## 2. Configuration

In [2]:
# Paths
BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / 'outputs' / 'processed'
VLM_DATASET_DIR = BASE_DIR / 'outputs' / 'vlm_dataset'

# Create output directories
VLM_DATASET_DIR.mkdir(parents=True, exist_ok=True)
(VLM_DATASET_DIR / 'images').mkdir(exist_ok=True)

# Load production summary
PRODUCTION_SUMMARY = PROCESSED_DIR / 'production_summary.json'

# Class information
CLASS_NAMES = {
    0: 'Cassava Bacterial Blight (CBB)',
    1: 'Cassava Brown Streak Disease (CBSD)',
    2: 'Cassava Green Mottle (CGM)',
    3: 'Cassava Mosaic Disease (CMD)',
    4: 'Healthy'
}

CLASS_NAMES_SHORT = {
    0: 'CBB',
    1: 'CBSD',
    2: 'CGM',
    3: 'CMD',
    4: 'Healthy'
}

# Disease descriptions for VLM prompts
CLASS_DESCRIPTIONS = {
    0: 'Cassava Bacterial Blight (CBB) is a bacterial disease causing water-soaked lesions, leaf wilting, and stem cankers.',
    1: 'Cassava Brown Streak Disease (CBSD) causes brown streaks on stems and necrosis in tubers, severely affecting yield.',
    2: 'Cassava Green Mottle (CGM) presents as green chlorotic mottling patterns on leaves with mild mosaic symptoms.',
    3: 'Cassava Mosaic Disease (CMD) is characterized by severe mosaic patterns, leaf distortion, and stunted growth.',
    4: 'Healthy cassava plants show uniform green leaves without disease symptoms, discoloration, or deformities.'
}

# Split ratios
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# Random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("✓ Configuration loaded")
print(f"\nOutput directory: {VLM_DATASET_DIR}")
print(f"Processed images: {PROCESSED_DIR / 'images'}")
print(f"\nSplit ratios:")
print(f"  Train: {TRAIN_RATIO*100:.0f}%")
print(f"  Val: {VAL_RATIO*100:.0f}%")
print(f"  Test: {TEST_RATIO*100:.0f}%")

✓ Configuration loaded

Output directory: /home/hounfodjidagba/learning/cassava/outputs/vlm_dataset
Processed images: /home/hounfodjidagba/learning/cassava/outputs/processed/images

Split ratios:
  Train: 80%
  Val: 10%
  Test: 10%


## 3. Load Production Results

In [3]:
# Load production summary
print("Loading production results...")
with open(PRODUCTION_SUMMARY, 'r') as f:
    prod_summary = json.load(f)

print(f"✓ Production summary loaded")
print(f"\nProduction Statistics:")
print(f"  Total processed: {prod_summary['total_images']:,}")
print(f"  Successful: {prod_summary['successful']:,} ({prod_summary['successful']/prod_summary['total_images']*100:.1f}%)")
print(f"  Primary: {prod_summary['primary']:,}")
print(f"  Fallback: {prod_summary['fallback']:,}")
print(f"  Failed: {prod_summary['failed']:,}")

print(f"\nBy Class:")
for class_label, stats in sorted(prod_summary['by_class'].items()):
    class_label = int(class_label)
    print(f"  {class_label} - {stats['name']:10s}: {stats['successful']:,} images")

Loading production results...
✓ Production summary loaded

Production Statistics:
  Total processed: 30,513
  Successful: 30,491 (99.9%)
  Primary: 28,995
  Fallback: 1,496
  Failed: 22

By Class:
  0 - CBB       : 1,087 images
  1 - CBSD      : 5,186 images
  2 - CGM       : 2,386 images
  3 - CMD       : 18,136 images
  4 - Healthy   : 3,696 images


## 4. Collect All Successfully Processed Images

In [4]:
# Collect all processed images with their metadata
print("Collecting processed images and metadata...\n")

all_samples = []

for class_label in sorted(CLASS_NAMES.keys()):
    class_name = CLASS_NAMES_SHORT[class_label]
    
    # Paths
    images_dir = PROCESSED_DIR / 'images' / f"{class_label}_{class_name}"
    metadata_dir = PROCESSED_DIR / 'metadata' / f"{class_label}_{class_name}"
    
    # Get all images
    image_files = sorted(images_dir.glob('*.png'))
    
    print(f"Class {class_label} ({class_name}): {len(image_files):,} images")
    
    for img_path in image_files:
        image_id = img_path.stem
        meta_path = metadata_dir / f"{image_id}.json"
        
        # Load metadata
        if meta_path.exists():
            with open(meta_path, 'r') as f:
                metadata = json.load(f)
        else:
            metadata = {}
        
        # Create sample entry
        sample = {
            'image_id': image_id,
            'image_path': str(img_path),
            'label': class_label,
            'class_name': CLASS_NAMES[class_label],
            'class_short': class_name,
            'description': CLASS_DESCRIPTIONS[class_label],
            'method': metadata.get('method', 'unknown'),
            'used_fallback': metadata.get('used_fallback', False),
            'source': metadata.get('source', 'unknown'),
            'disease': metadata.get('disease', CLASS_NAMES[class_label])
        }
        
        all_samples.append(sample)

print(f"\n✓ Total samples collected: {len(all_samples):,}")

# Convert to DataFrame
df = pd.DataFrame(all_samples)
print(f"✓ DataFrame created: {df.shape}")


Class 0 (CBB): 1,087 images
Class 1 (CBSD): 5,186 images
Class 2 (CGM): 2,386 images
Class 3 (CMD): 18,136 images
Class 4 (Healthy): 3,696 images

✓ Total samples collected: 30,491
✓ DataFrame created: (30491, 10)


## 5. Create Stratified Train/Val/Test Splits

In [5]:
print("="*70)
print("CREATING STRATIFIED SPLITS")
print("="*70)

# First split: train vs (val+test)
train_df, temp_df = train_test_split(
    df,
    test_size=(VAL_RATIO + TEST_RATIO),
    stratify=df['label'],
    random_state=RANDOM_SEED
)

# Second split: val vs test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,  # 50/50 split of the 20%
    stratify=temp_df['label'],
    random_state=RANDOM_SEED
)

print(f"\nSplit sizes:")
print(f"  Train: {len(train_df):,} ({len(train_df)/len(df)*100:.1f}%)")
print(f"  Val:   {len(val_df):,} ({len(val_df)/len(df)*100:.1f}%)")
print(f"  Test:  {len(test_df):,} ({len(test_df)/len(df)*100:.1f}%)")
print(f"  Total: {len(df):,}")

# Verify stratification
print(f"\nClass distribution verification:")
print(f"\n{'Class':<15} {'Train':<10} {'Val':<10} {'Test':<10} {'Total':<10}")
print("-" * 60)

for class_label in sorted(CLASS_NAMES.keys()):
    class_name = CLASS_NAMES_SHORT[class_label]
    train_count = len(train_df[train_df['label'] == class_label])
    val_count = len(val_df[val_df['label'] == class_label])
    test_count = len(test_df[test_df['label'] == class_label])
    total = train_count + val_count + test_count
    
    print(f"{class_name:<15} {train_count:<10} {val_count:<10} {test_count:<10} {total:<10}")

print("\n✓ Stratified splits created successfully")

CREATING STRATIFIED SPLITS

Split sizes:
  Train: 24,392 (80.0%)
  Val:   3,049 (10.0%)
  Test:  3,050 (10.0%)
  Total: 30,491

Class distribution verification:

Class           Train      Val        Test       Total     
------------------------------------------------------------
CBB             869        109        109        1087      
CBSD            4149       518        519        5186      
CGM             1909       239        238        2386      
CMD             14508      1814       1814       18136     
Healthy         2957       369        370        3696      

✓ Stratified splits created successfully


## 6. Generate Qwen2.5-VL Format (JSONL)

In [6]:
def create_vlm_entry(sample, split='train'):
    """
    Create a VLM training entry in Qwen2.5-VL format.
    
    Format:
    {
        "id": "unique_id",
        "image": "path/to/image.png",
        "conversations": [
            {
                "from": "human",
                "value": "<image>\nWhat disease does this cassava plant have?"
            },
            {
                "from": "gpt",
                "value": "This cassava plant has..."
            }
        ]
    }
    """
    image_id = sample['image_id']
    class_name = sample['class_name']
    class_short = sample['class_short']
    description = sample['description']
    
    # Relative path for portability
    rel_image_path = f"images/{class_short}/{image_id}.png"
    
    # Create conversation
    entry = {
        "id": f"{split}_{image_id}",
        "image": rel_image_path,
        "conversations": [
            {
                "from": "human",
                "value": "<image>\nAnalyze this cassava leaf image and identify any disease present. Provide the disease name and a brief description."
            },
            {
                "from": "gpt",
                "value": f"Disease: {class_name}\n\n{description}"
            }
        ],
        "metadata": {
            "label": int(sample['label']),
            "class_short": class_short,
            "method": sample['method'],
            "used_fallback": sample['used_fallback'],
            "source": sample['source']
        }
    }
    
    return entry

print("✓ VLM entry generator defined")

✓ VLM entry generator defined


In [7]:
print("="*70)
print("GENERATING VLM DATASET FILES")
print("="*70)

# Generate JSONL files for each split
splits = [
    ('train', train_df),
    ('val', val_df),
    ('test', test_df)
]

for split_name, split_df in splits:
    print(f"\nGenerating {split_name}.jsonl...")
    
    jsonl_path = VLM_DATASET_DIR / f"{split_name}.jsonl"
    
    with open(jsonl_path, 'w') as f:
        for idx, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"{split_name}"):
            entry = create_vlm_entry(row, split=split_name)
            f.write(json.dumps(entry) + '\n')
    
    print(f"  ✓ {split_name}.jsonl saved: {len(split_df):,} entries")

print("\n✓ All JSONL files generated")

GENERATING VLM DATASET FILES

Generating train.jsonl...


train:   0%|          | 0/24392 [00:00<?, ?it/s]

  ✓ train.jsonl saved: 24,392 entries

Generating val.jsonl...


val:   0%|          | 0/3049 [00:00<?, ?it/s]

  ✓ val.jsonl saved: 3,049 entries

Generating test.jsonl...


test:   0%|          | 0/3050 [00:00<?, ?it/s]

  ✓ test.jsonl saved: 3,050 entries

✓ All JSONL files generated


## 7. Organize Images for VLM Dataset

In [8]:
print("="*70)
print("ORGANIZING IMAGES")
print("="*70)

# Create class subdirectories
for class_label in CLASS_NAMES.keys():
    class_short = CLASS_NAMES_SHORT[class_label]
    (VLM_DATASET_DIR / 'images' / class_short).mkdir(parents=True, exist_ok=True)

# Copy images (or create symlinks to save space)
print("\nCopying/linking images...")
print("Note: Creating symlinks to save disk space\n")

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Organizing"):
    src_path = Path(row['image_path'])
    class_short = row['class_short']
    image_id = row['image_id']
    
    dst_path = VLM_DATASET_DIR / 'images' / class_short / f"{image_id}.png"
    
    # Create symlink if it doesn't exist
    if not dst_path.exists():
        try:
            dst_path.symlink_to(src_path.resolve())
        except:
            # Fallback to copy if symlink fails
            shutil.copy2(src_path, dst_path)

print("\n✓ Images organized")

# Verify
print("\nVerifying image organization:")
for class_label in sorted(CLASS_NAMES.keys()):
    class_short = CLASS_NAMES_SHORT[class_label]
    img_dir = VLM_DATASET_DIR / 'images' / class_short
    count = len(list(img_dir.glob('*.png')))
    print(f"  {class_short}: {count:,} images")

total_images = sum(len(list((VLM_DATASET_DIR / 'images' / c).glob('*.png'))) 
                  for c in CLASS_NAMES_SHORT.values())
print(f"\n✓ Total: {total_images:,} images")

ORGANIZING IMAGES

Copying/linking images...
Note: Creating symlinks to save disk space



Organizing:   0%|          | 0/30491 [00:00<?, ?it/s]


✓ Images organized

Verifying image organization:
  CBB: 1,087 images
  CBSD: 5,186 images
  CGM: 2,386 images
  CMD: 18,136 images
  Healthy: 3,696 images

✓ Total: 30,491 images


## 8. Create Dataset Metadata

In [9]:
# Create comprehensive dataset info
dataset_info = {
    'name': 'CassavaVLM',
    'version': '1.0',
    'description': 'Cassava disease detection dataset for Vision-Language Models',
    'created_date': datetime.now().isoformat(),
    'total_images': len(df),
    'image_size': [448, 448],
    'num_classes': len(CLASS_NAMES),
    
    'splits': {
        'train': {
            'num_samples': len(train_df),
            'percentage': len(train_df)/len(df)*100
        },
        'val': {
            'num_samples': len(val_df),
            'percentage': len(val_df)/len(df)*100
        },
        'test': {
            'num_samples': len(test_df),
            'percentage': len(test_df)/len(df)*100
        }
    },
    
    'class_distribution': {},
    
    'preprocessing': {
        'primary_pipeline': f"{prod_summary['primary']} images ({prod_summary['primary']/prod_summary['total_images']*100:.1f}%)",
        'fallback_pipeline': f"{prod_summary['fallback']} images ({prod_summary['fallback']/prod_summary['total_images']*100:.1f}%)",
        'success_rate': f"{prod_summary['successful']/prod_summary['total_images']*100:.2f}%"
    },
    
    'format': 'Qwen2.5-VL JSONL',
    'files': [
        'train.jsonl',
        'val.jsonl',
        'test.jsonl',
        'dataset_info.json',
        'class_mapping.json'
    ]
}

# Add class distribution
for class_label in sorted(CLASS_NAMES.keys()):
    class_name = CLASS_NAMES[class_label]
    class_short = CLASS_NAMES_SHORT[class_label]
    class_count = len(df[df['label'] == class_label])
    
    dataset_info['class_distribution'][class_short] = {
        'label': int(class_label),
        'name': class_name,
        'count': int(class_count),
        'percentage': float(class_count / len(df) * 100),
        'description': CLASS_DESCRIPTIONS[class_label],
        'train': int(len(train_df[train_df['label'] == class_label])),
        'val': int(len(val_df[val_df['label'] == class_label])),
        'test': int(len(test_df[test_df['label'] == class_label]))
    }

# Save dataset info
dataset_info_path = VLM_DATASET_DIR / 'dataset_info.json'
with open(dataset_info_path, 'w') as f:
    json.dump(dataset_info, f, indent=2)

print("✓ dataset_info.json created")

# Create class mapping
class_mapping = {
    'classes': CLASS_NAMES,
    'classes_short': CLASS_NAMES_SHORT,
    'descriptions': CLASS_DESCRIPTIONS,
    'num_classes': len(CLASS_NAMES)
}

class_mapping_path = VLM_DATASET_DIR / 'class_mapping.json'
with open(class_mapping_path, 'w') as f:
    json.dump(class_mapping, f, indent=2)

print("✓ class_mapping.json created")

✓ dataset_info.json created
✓ class_mapping.json created


## 9. Final Summary & Validation

In [10]:
print("="*70)
print("VLM DATASET PREPARATION COMPLETE")
print("="*70)

print(f"\n📁 Dataset Location: {VLM_DATASET_DIR}")
print(f"\n📊 Dataset Statistics:")
print(f"  Total images: {len(df):,}")
print(f"  Train: {len(train_df):,} ({len(train_df)/len(df)*100:.1f}%)")
print(f"  Val: {len(val_df):,} ({len(val_df)/len(df)*100:.1f}%)")
print(f"  Test: {len(test_df):,} ({len(test_df)/len(df)*100:.1f}%)")

print(f"\n📋 Files Generated:")
files = [
    ('train.jsonl', VLM_DATASET_DIR / 'train.jsonl'),
    ('val.jsonl', VLM_DATASET_DIR / 'val.jsonl'),
    ('test.jsonl', VLM_DATASET_DIR / 'test.jsonl'),
    ('dataset_info.json', VLM_DATASET_DIR / 'dataset_info.json'),
    ('class_mapping.json', VLM_DATASET_DIR / 'class_mapping.json')
]

for fname, fpath in files:
    size_mb = fpath.stat().st_size / (1024*1024) if fpath.exists() else 0
    status = "✓" if fpath.exists() else "✗"
    print(f"  {status} {fname:25s} ({size_mb:.2f} MB)")

print(f"\n🎯 Class Distribution:")
for class_short, info in dataset_info['class_distribution'].items():
    print(f"  {class_short:10s}: {info['count']:,} images ({info['percentage']:.1f}%)")
    print(f"             Train: {info['train']:,}, Val: {info['val']}, Test: {info['test']}")

print(f"\n✅ DATASET READY FOR VLM TRAINING")
print("="*70)

print(f"\n📝 Next Steps:")
print(f"  1. Review dataset_info.json for details")
print(f"  2. Verify a few samples from train.jsonl")
print(f"  3. Configure your VLM training script")
print(f"  4. Start training with Qwen2.5-VL or similar model")
print(f"\n💡 Training Tips:")
print(f"  - Use the provided class descriptions in prompts")
print(f"  - Consider data augmentation for minority classes (CBB, CGM)")
print(f"  - Monitor validation metrics across all classes")
print(f"  - Test set is held out for final evaluation")

VLM DATASET PREPARATION COMPLETE

📁 Dataset Location: /home/hounfodjidagba/learning/cassava/outputs/vlm_dataset

📊 Dataset Statistics:
  Total images: 30,491
  Train: 24,392 (80.0%)
  Val: 3,049 (10.0%)
  Test: 3,050 (10.0%)

📋 Files Generated:
  ✓ train.jsonl               (12.79 MB)
  ✓ val.jsonl                 (1.59 MB)
  ✓ test.jsonl                (1.60 MB)
  ✓ dataset_info.json         (0.00 MB)
  ✓ class_mapping.json        (0.00 MB)

🎯 Class Distribution:
  CBB       : 1,087 images (3.6%)
             Train: 869, Val: 109, Test: 109
  CBSD      : 5,186 images (17.0%)
             Train: 4,149, Val: 518, Test: 519
  CGM       : 2,386 images (7.8%)
             Train: 1,909, Val: 239, Test: 238
  CMD       : 18,136 images (59.5%)
             Train: 14,508, Val: 1814, Test: 1814
  Healthy   : 3,696 images (12.1%)
             Train: 2,957, Val: 369, Test: 370

✅ DATASET READY FOR VLM TRAINING

📝 Next Steps:
  1. Review dataset_info.json for details
  2. Verify a few samples from